<a href="https://colab.research.google.com/github/amiralito/BUSCO_colab/blob/main/BUSCO_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BUSCO — batch completeness assessment on Google Colab

[BUSCO](https://busco.ezlab.org) scores the completeness of a genome assembly, an annotation (predicted proteome) or a transcriptome against a lineage-specific set of near-universal single-copy orthologs from OrthoDB. This notebook runs BUSCO over **one or many FASTA files** against **one or more lineages**, archives every run directory as `.tar.gz`, keeps each run's short summary and full table, and compiles everything into a single score table, a stacked-bar plot and a BUSCO × sample status matrix.

It is the Colab counterpart of the HPC BUSCO scripts: same per-sample loop over a list of lineages, compressed run directories, consolidated `busco_summary.tsv` with `NA` rows for failed samples — plus Drive-cached lineage datasets and resumable batches.

**How to use:** a **CPU runtime is enough** (BUSCO does not use the GPU). Fill the form fields and run each cell top to bottom. **The code is hidden** — to edit a cell, open its **menu (vertical dots) > Show code**.

| Mode | Input | Engine | Speed on Colab |
|---|---|---|---|
| `proteins` | predicted proteome | HMMER | minutes per proteome |
| `genome` | assembly | miniprot (or MetaEuk) → HMMER | tens of minutes to hours |
| `transcriptome` | assembled transcripts | MetaEuk → HMMER | tens of minutes to hours |

Mount Drive in cell 3 to read inputs from a folder, cache lineage datasets and keep results across sessions.

In [ ]:
#@title 1 · Check runtime — CPU, RAM, disk { display-mode: "form" }
#@markdown BUSCO is CPU-only; every core is used by default. Colab Pro / High-RAM runtimes give more cores.
import os, sys, shutil, subprocess

print("Python :", sys.version.split()[0])
print("CPUs   :", os.cpu_count())
try:
    kb = int(open("/proc/meminfo").read().split("MemTotal:")[1].split()[0])
    print(f"RAM    : {kb / 1e6:.1f} GB")
except Exception:
    pass
free = shutil.disk_usage("/content" if os.path.isdir("/content") else "/").free
print(f"Disk   : {free / 1e9:.0f} GB free on /content")
gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
if gpu:
    print("GPU    :", gpu.splitlines()[0], "\n         (unused by BUSCO - a CPU runtime saves GPU quota)")
if (os.cpu_count() or 1) <= 2:
    print("\nFree-tier runtime (2 cores): proteins mode is quick; genome/transcriptome mode on large "
          "assemblies is slow - prefer HPC for those.")

In [ ]:
#@title 2 · Install BUSCO + gene predictors { display-mode: "form" }
#@markdown BUSCO from source (GitLab) with HMMER for every mode, plus **miniprot** (genome mode, default for eukaryotes),
#@markdown **MetaEuk** (transcriptome mode; optional genome predictor), **BBTools** (assembly statistics),
#@markdown **Prodigal** (prokaryote genomes) and **pigz** (parallel compression). ~1-2 min; re-running is a no-op.
BUSCO_VERSION = "latest"  #@param ["latest", "6.1.0", "6.0.0", "5.8.3"] {allow-input: true}
#@markdown `latest` = BUSCO's current default branch. The `*_odb12.2` datasets need BUSCO >= 6.1; with 5.8.x / 6.0.x use the `*_odb12` names.
import os, sys, shutil, subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"failed: {cmd}")

if not all(shutil.which(t) for t in ("hmmsearch", "pigz", "prodigal", "stats.sh")):
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y hmmer pigz prodigal bbmap > /dev/null")
    for t in ("stats.sh", "statswrapper.sh"):
        if os.path.exists(f"/usr/share/bbmap/{t}") and not shutil.which(t):
            sh(f"ln -sf /usr/share/bbmap/{t} /usr/local/bin/{t}")

if not shutil.which("miniprot"):
    sh("rm -rf /content/miniprot && git clone -q --depth 1 https://github.com/lh3/miniprot /content/miniprot")
    sh("make -s -C /content/miniprot -j$(nproc) && cp /content/miniprot/miniprot /usr/local/bin/")

if not shutil.which("metaeuk"):
    simd = "avx2" if "avx2" in open("/proc/cpuinfo").read() else "sse41"
    sh(f"cd /content && wget -q https://mmseqs.com/metaeuk/metaeuk-linux-{simd}.tar.gz -O metaeuk.tar.gz "
       "&& tar xzf metaeuk.tar.gz && cp metaeuk/bin/metaeuk /usr/local/bin/ && rm -rf metaeuk metaeuk.tar.gz")

def busco_version():
    r = subprocess.run(["busco", "--version"], capture_output=True, text=True)
    return r.stdout.strip().replace("BUSCO ", "") if r.returncode == 0 else None

want = None if BUSCO_VERSION == "latest" else BUSCO_VERSION.strip()
have = busco_version() if shutil.which("busco") else None
if have is None or (want and have != want):
    branch = f"--branch {want} " if want else ""
    sh(f"rm -rf /content/busco && git clone -q --depth 1 {branch}https://gitlab.com/ezlab/busco.git /content/busco")
    sh(f"{sys.executable} -m pip install -q /content/busco biopython pandas requests")

def ver(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    return out[0] if out else "?"

TOOL_VERSIONS = {
    "busco": busco_version(),
    "hmmer": ver("hmmsearch -h | sed -n 2p").lstrip("# "),
    "miniprot": ver("miniprot --version"),
    "metaeuk": ver("metaeuk version"),
    "prodigal": ver("prodigal -v 2>&1 | grep -i prodigal"),
    "bbtools": "installed" if shutil.which("stats.sh") else "missing",
}
for k, v in TOOL_VERSIONS.items():
    print(f"{k:<9}: {v}")
assert TOOL_VERSIONS["busco"], "BUSCO did not install - see the output above."

In [ ]:
#@title 3 · Storage — mount Drive & start a run folder { display-mode: "form" }
#@markdown Mount Google Drive so inputs, lineage datasets and results persist across sessions.
#@markdown Every run gets its own timestamped folder under `WORKDIR/runs/`, so nothing is overwritten.
MOUNT_DRIVE = True  #@param {type:"boolean"}
WORKDIR = "/content/drive/MyDrive/busco_runs"  #@param {type:"string"}
RUN_TAG = ""  #@param {type:"string"}
TIMEZONE = "Europe/London"  #@param {type:"string"}
#@markdown `RUN_TAG` is an optional suffix for the folder name (e.g. `oomycetes`). `TIMEZONE` only affects
#@markdown the timestamp; the Colab clock itself is UTC.
RESUME_RUN = ""  #@param {type:"string"}
#@markdown To **continue an interrupted batch** in a new session, paste the existing run folder name here
#@markdown (e.g. `run_20261015-120000_oomycetes`); with `SKIP_DONE` (cell 4) finished runs are skipped.
import os, re
from datetime import datetime, timezone as _tz

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
elif WORKDIR.startswith("/content/drive"):
    WORKDIR = "/content/work"

try:
    from zoneinfo import ZoneInfo
    NOW = datetime.now(ZoneInfo(TIMEZONE))
except Exception:
    NOW, TIMEZONE = datetime.now(_tz.utc), "UTC"

tag = re.sub(r"[^A-Za-z0-9._-]+", "_", RUN_TAG.strip()).strip("_")
if RESUME_RUN.strip():
    RUN_ID = os.path.basename(RESUME_RUN.strip().rstrip("/"))
    assert os.path.isdir(os.path.join(WORKDIR, "runs", RUN_ID)), \
        f"No run folder {RUN_ID} under {os.path.join(WORKDIR, 'runs')}"
    print("Resuming existing run folder.")
else:
    RUN_ID = NOW.strftime("run_%Y%m%d-%H%M%S") + (f"_{tag}" if tag else "")
RUNDIR = os.path.join(WORKDIR, "runs", RUN_ID)
LINEAGE_CACHE = os.path.join(WORKDIR, "busco_lineages")   # shared across runs, downloaded once
os.makedirs(RUNDIR, exist_ok=True)
os.makedirs(LINEAGE_CACHE, exist_ok=True)

print("Working directory:", WORKDIR)
print("Run folder       :", RUNDIR, f"({TIMEZONE})")
print("Lineage cache    :", LINEAGE_CACHE)
print("\nRe-run this cell to start a new run folder; re-running cells 4-10 alone keeps writing to this one.")

In [ ]:
#@title 4 · BUSCO parameters & input { display-mode: "form" }
#@markdown **Mode** — `proteins` for a predicted proteome (how complete the *annotation* is), `genome` for an
#@markdown assembly (the completeness *ceiling*, independent of annotation), `transcriptome` for assembled transcripts.
MODE = "proteins"  #@param ["proteins", "genome", "transcriptome"]
GENOME_PREDICTOR = "miniprot"  #@param ["miniprot", "metaeuk"]
#@markdown `GENOME_PREDICTOR` applies to eukaryotic lineages in genome mode (miniprot is BUSCO's default and much faster).
#@markdown ---
#@markdown **Lineage(s)** — pick one, or type several separated by commas, e.g. `stramenopiles_odb12.2, oomycota_odb12.2`.
#@markdown Every input is scored against each. A full path to a lineage folder also works. List all with `busco --list-datasets`.
LINEAGES = "eudicotyledons_odb12.2"  #@param ["eukaryota_odb12.2", "viridiplantae_odb12.2", "embryophyta_odb12.2", "eudicotyledons_odb12.2", "solanales_odb12.2", "brassicales_odb12.2", "fabales_odb12.2", "liliopsida_odb12.2", "poales_odb12.2", "chlorophyta_odb12.2", "stramenopiles_odb12.2", "oomycota_odb12.2", "bacillariophyta_odb12.2", "fungi_odb12.2", "ascomycota_odb12.2", "basidiomycota_odb12.2", "sordariomycetes_odb12.2", "eurotiomycetes_odb12.2", "saccharomycetes_odb12.2", "metazoa_odb12.2", "arthropoda_odb12.2", "insecta_odb12.2", "hemiptera_odb12.2", "diptera_odb12.2", "hymenoptera_odb12.2", "lepidoptera_odb12.2", "nematoda_odb12.2", "mollusca_odb12.2", "vertebrata_odb12.2", "actinopterygii_odb12.2", "aves_odb12.2", "mammalia_odb12.2", "alveolata_odb12.2", "apicomplexa_odb12.2"] {allow-input: true}
LINEAGE_DIR = ""  #@param {type:"string"}
#@markdown Optional Drive folder of pre-downloaded lineage folders (e.g. a copy of the HPC `busco_lineages/`); used before downloading.
#@markdown ---
#@markdown **Input** — FASTA files (`.fa/.fasta/.fna/.faa/.fas/.pep`..., optionally `.gz`). Each file is one sample, named after the file.
INPUT_SOURCE = "drive_dir"  #@param ["drive_dir", "drive_file", "upload", "url"]
INPUT_DIR = "/content/drive/MyDrive/busco_inputs"  #@param {type:"string"}
RECURSIVE = True  #@param {type:"boolean"}
INPUT_PATH = ""  #@param {type:"string"}
INPUT_URL = ""  #@param {type:"string"}
#@markdown `INPUT_DIR` (+ `RECURSIVE`) for `drive_dir`, `INPUT_PATH` for `drive_file`, `INPUT_URL` for `url` (comma-separate several URLs).
#@markdown ---
#@markdown **Run control**
CPUS = 0  #@param {type:"integer"}
#@markdown `CPUS = 0` uses every core.
SKIP_DONE = True  #@param {type:"boolean"}
STOP_ON_ERROR = False  #@param {type:"boolean"}
#@markdown `SKIP_DONE` reuses runs already summarised in this run folder; with `STOP_ON_ERROR` off a failed sample gets an `NA` row and the batch carries on.
KEEP_LOCAL_RUNS = False  #@param {type:"boolean"}
#@markdown Each finished run is archived to the run folder at once; the local copy in `/content/busco_runs` is deleted unless `KEEP_LOCAL_RUNS` is on.
EXTRA_ARGS = ""  #@param {type:"string"}
#@markdown Passed verbatim to every `busco` call, e.g. `--limit 5` or `--contig_break 20`.
import os, re

LINEAGE_LIST = [x for x in re.split(r"[,\s]+", LINEAGES.strip()) if x]
assert LINEAGE_LIST, "Set at least one lineage."
dup = {x for x in LINEAGE_LIST if LINEAGE_LIST.count(x) > 1}
assert not dup, f"Lineage listed twice: {sorted(dup)}"
NCPU = CPUS if CPUS > 0 else (os.cpu_count() or 2)
MODE_TAG = {"proteins": "prot", "genome": "genome", "transcriptome": "tx"}[MODE]

if not MOUNT_DRIVE:
    for v in (INPUT_DIR, INPUT_PATH, LINEAGE_DIR):
        if INPUT_SOURCE.startswith("drive") and v.startswith("/content/drive"):
            print(f"Warning: {v} is on Drive but MOUNT_DRIVE is off (cell 3).")

print("mode     :", MODE + (f" ({GENOME_PREDICTOR})" if MODE == "genome" else ""))
print("lineages :", ", ".join(LINEAGE_LIST))
print("input    :", INPUT_SOURCE, "->", {"drive_dir": INPUT_DIR, "drive_file": INPUT_PATH,
                                         "url": INPUT_URL, "upload": "(file picker)"}[INPUT_SOURCE])
print("cpus     :", NCPU)

In [ ]:
#@title 5 · Fetch lineage dataset(s) { display-mode: "form" }
#@markdown Each lineage is taken from, in order: this session's copy, a folder path typed in `LINEAGES`, `LINEAGE_DIR`,
#@markdown the Drive cache (`WORKDIR/busco_lineages/<lineage>.tar`), or a fresh download from busco-data.ezlab.org,
#@markdown which is then cached on Drive so later sessions skip it. All runs read the local copy (`/content/busco_downloads`).
import os, shutil, subprocess, tarfile
import pandas as pd
from IPython.display import display

DL = "/content/busco_downloads"
LIN_LOCAL = os.path.join(DL, "lineages")
os.makedirs(LIN_LOCAL, exist_ok=True)

def is_lineage(d):
    return os.path.isfile(os.path.join(d, "dataset.cfg"))

def read_cfg(d):
    cfg = {}
    for line in open(os.path.join(d, "dataset.cfg")):
        if "=" in line:
            k, v = line.strip().split("=", 1)
            cfg[k] = v
    return cfg

def untar(path, dest):
    with tarfile.open(path) as t:
        try:
            t.extractall(dest, filter="data")
        except TypeError:                      # Python < 3.12
            t.extractall(dest)

LINEAGE_INFO = {}
for entry in LINEAGE_LIST:
    name = os.path.basename(entry.rstrip("/"))
    local = os.path.join(LIN_LOCAL, name)
    cached = os.path.join(LINEAGE_CACHE, name + ".tar")
    in_dir = os.path.join(LINEAGE_DIR.strip(), name) if LINEAGE_DIR.strip() else ""
    if is_lineage(local):
        src = "session copy"
    elif "/" in entry and is_lineage(entry):
        shutil.copytree(entry, local); src = "folder path"
    elif in_dir and is_lineage(in_dir):
        shutil.copytree(in_dir, local); src = "LINEAGE_DIR"
    elif os.path.exists(cached):
        untar(cached, LIN_LOCAL); src = "Drive cache"
    else:
        assert "/" not in entry, f"Not a BUSCO lineage folder (no dataset.cfg): {entry}"
        print(f"downloading {name} ...")
        r = subprocess.run(["busco", "--download_path", DL, "--download", name], capture_output=True, text=True)
        errs = [l for l in (r.stdout + r.stderr).splitlines() if "ERROR" in l]
        assert is_lineage(local), (f"Could not download '{name}':\n" + "\n".join(errs) +
                                   "\nCheck the exact name with:  !busco --list-datasets")
        src = "downloaded"
        if MOUNT_DRIVE:
            tmp = os.path.join("/content", name + ".tar")
            with tarfile.open(tmp, "w") as t:
                t.add(local, arcname=name)
            shutil.move(tmp, cached)
            src += " + cached on Drive"
    assert is_lineage(local), f"{name}: dataset.cfg missing after staging"
    cfg = read_cfg(local)
    LINEAGE_INFO[name] = {"path": local, "source": src, **cfg}

display(pd.DataFrame([{"lineage": k, "domain": v.get("domain"), "BUSCOs": v.get("number_of_BUSCOs"),
                       "species": v.get("number_of_species"), "created": v.get("creation_date"),
                       "source": v["source"]} for k, v in LINEAGE_INFO.items()]).to_string(index=False))

In [ ]:
#@title 6 · Collect input FASTA(s) { display-mode: "form" }
#@markdown Stages a plain (uncompressed) local copy of every input and checks that its alphabet matches `MODE`.
import os, re, gzip, shutil, hashlib, subprocess

STAGE = "/content/busco_inputs"; os.makedirs(STAGE, exist_ok=True)
FASTA_EXT = (".fa", ".fasta", ".fna", ".faa", ".fas", ".fsa", ".pep", ".aa", ".ffn", ".cds",
             ".mrna", ".transcripts", ".seq")

def _plain(name):
    n = name.lower()
    return n[:-3] if n.endswith(".gz") else n

def is_fasta(fn):
    return _plain(fn).endswith(FASTA_EXT)

def base_of(p):
    b = os.path.basename(p)
    b = b[:-3] if b.lower().endswith(".gz") else b
    root, ext = os.path.splitext(b)
    b = root if ext.lower() in FASTA_EXT else b
    return re.sub(r"[^A-Za-z0-9._-]+", "_", b).strip("_")

def stage_local(src, base):
    # salted with the source path so same-named files in different subfolders cannot collide
    salt = hashlib.md5(os.path.abspath(src).encode()).hexdigest()[:6]
    out = os.path.join(STAGE, f"{base}__{salt}.fasta")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return out
    opener = gzip.open if src.lower().endswith(".gz") else open
    with opener(src, "rb") as fi, open(out + ".part", "wb") as fo:
        shutil.copyfileobj(fi, fo, 16 << 20)
    os.replace(out + ".part", out)
    return out

def sniff(path, nbytes=2_000_000):
    # (first char, nucleotide fraction of residues in the first ~2 MB)
    with open(path, "rb") as f:
        head = f.read(nbytes).decode("ascii", "ignore")
    seq = "".join(l for l in head.splitlines() if l and not l.startswith(">")).upper()
    nuc = sum(seq.count(c) for c in "ACGTUN") / max(len(seq), 1)
    return head[:1], nuc

raw = []
if INPUT_SOURCE == "drive_dir":
    assert os.path.isdir(INPUT_DIR), f"Not a directory: {INPUT_DIR}"
    if RECURSIVE:
        for root, _, fnames in os.walk(INPUT_DIR):
            raw += [os.path.join(root, f) for f in fnames if is_fasta(f)]
    else:
        raw = [os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR)
               if is_fasta(f) and os.path.isfile(os.path.join(INPUT_DIR, f))]
    assert raw, f"No FASTA files found under {INPUT_DIR}"
elif INPUT_SOURCE == "drive_file":
    assert os.path.isfile(INPUT_PATH), f"Not found: {INPUT_PATH}"
    raw = [INPUT_PATH]
elif INPUT_SOURCE == "url":
    urls = [u.strip() for u in INPUT_URL.split(",") if u.strip()]
    assert urls, "Set INPUT_URL in cell 4."
    dl_dir = os.path.join(STAGE, "_downloads"); os.makedirs(dl_dir, exist_ok=True)
    for u in urls:
        dst = os.path.join(dl_dir, u.split("/")[-1].split("?")[0])
        subprocess.run(["wget", "-q", "-O", dst, u], check=True)
        raw.append(dst)
elif INPUT_SOURCE == "upload":
    from google.colab import files
    up_dir = os.path.join(STAGE, "_uploads"); os.makedirs(up_dir, exist_ok=True)
    for name in files.upload():
        d = os.path.join(up_dir, name); os.replace(name, d); raw.append(d)

INPUTS = []
for p in sorted(raw):
    b = base_of(p)
    INPUTS.append({"sample": b, "src": p, "path": stage_local(p, b)})
names = [j["sample"] for j in INPUTS]
dupes = sorted({n for n in names if names.count(n) > 1})
assert not dupes, f"Sample names must be unique - these appear more than once: {dupes}"

want_nuc = MODE != "proteins"
bad = []
print(f"{len(INPUTS)} input(s) staged for {MODE} mode:")
for j in INPUTS:
    first, nuc = sniff(j["path"])
    j["n_seqs"] = int(subprocess.run(["grep", "-c", "^>", j["path"]], capture_output=True, text=True).stdout or 0)
    j["mb"] = os.path.getsize(j["path"]) / 1e6
    kind = "nucleotide" if nuc > 0.9 else "protein"
    flag = ""
    if first != ">":
        flag = "  <- not FASTA"; bad.append(j["sample"])
    elif (kind == "nucleotide") != want_nuc:
        flag = f"  <- looks {kind}, mode is {MODE}"; bad.append(j["sample"])
    print(f"   {j['sample']:<42}{j['n_seqs']:>9,} seqs {j['mb']:9.1f} MB  {kind}{flag}")
assert not bad, f"{len(bad)} input(s) do not match MODE={MODE}: {bad}"

In [ ]:
#@title 7 · Run BUSCO on all input(s) × lineage(s) { display-mode: "form" }
#@markdown BUSCO runs locally in `/content/busco_runs` (it uses symlinks, which Drive does not support). After **each** run
#@markdown the short summary (`.txt` + `.json`) and `full_table.tsv` are copied to the run folder, the BUSCO directory is
#@markdown archived as `.tar.gz`, and `busco_summary.tsv` is rewritten — an interrupted batch loses at most the run in progress.
import os, re, csv, json, glob, gzip, shlex, shutil, subprocess, tarfile, time
from datetime import datetime

LOCAL_RUNS = "/content/busco_runs"; os.makedirs(LOCAL_RUNS, exist_ok=True)
SUB = {k: os.path.join(RUNDIR, k) for k in ("summaries", "full_tables", "archives", "logs")}
for d in SUB.values():
    os.makedirs(d, exist_ok=True)
TABLE = os.path.join(RUNDIR, "busco_summary.tsv")
COLS = ["sample", "mode", "lineage", "status", "C%", "S%", "D%", "F%", "M%", "n",
        "complete", "single", "duplicated", "fragmented", "missing", "stop_codon%", "one_line",
        "scaffolds", "contigs", "total_length", "scaffold_N50", "contig_N50", "gaps%",
        "busco_version", "lineage_date", "minutes", "run", "input"]
KEY = lambda r: (r["sample"], r["mode"], r["lineage"])
safe = lambda s: re.sub(r"[^A-Za-z0-9._-]+", "_", s)

def load_table():
    if not os.path.exists(TABLE):
        return {}
    with open(TABLE) as f:
        return {KEY(r): r for r in csv.DictReader(f, delimiter="\t")}

def save_table(rows):
    tmp = "/content/busco_summary.tsv.part"
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=COLS, delimiter="\t", extrasaction="ignore", restval="NA")
        w.writeheader()
        w.writerows(rows[k] for k in sorted(rows))
    shutil.copy(tmp, TABLE)

def row_from_json(path, sample, lineage, run, inp, minutes="NA"):
    js = json.load(open(path))
    res, met = js["results"], js.get("metrics", {})
    g = lambda d, k: d.get(k, "NA")
    return {"sample": sample, "mode": MODE, "lineage": lineage, "status": "ok",
            "C%": g(res, "Complete percentage"), "S%": g(res, "Single copy percentage"),
            "D%": g(res, "Multi copy percentage"), "F%": g(res, "Fragmented percentage"),
            "M%": g(res, "Missing percentage"), "n": g(res, "n_markers"),
            "complete": g(res, "Complete BUSCOs"), "single": g(res, "Single copy BUSCOs"),
            "duplicated": g(res, "Multi copy BUSCOs"), "fragmented": g(res, "Fragmented BUSCOs"),
            "missing": g(res, "Missing BUSCOs"), "stop_codon%": g(res, "internal_stop_codon_percent"),
            "one_line": g(res, "one_line_summary"),
            "scaffolds": g(met, "Number of scaffolds"), "contigs": g(met, "Number of contigs"),
            "total_length": g(met, "Total length"), "scaffold_N50": g(met, "Scaffold N50"),
            "contig_N50": g(met, "Contigs N50"), "gaps%": str(g(met, "Percent gaps")).rstrip("%"),
            "busco_version": js.get("versions", {}).get("busco", "NA"),
            "lineage_date": js.get("lineage_dataset", {}).get("creation_date", "NA"),
            "minutes": minutes, "run": run, "input": inp}

ECHO = ("ERROR", "WARNING", "Results:", "Running BUSCO using", "Running Miniprot", "Running Metaeuk",
        "Running Prodigal", "Running hmmsearch", "BUSCO analysis done")

def run_logged(cmd, log_path):
    with open(log_path, "w") as log:
        log.write("$ " + shlex.join(cmd) + "\n\n")
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line)
            if any(k in line for k in ECHO):
                print("   " + line.rstrip()[:160])
        return p.wait()

def archive(run_dir, dest):
    tmp = os.path.join("/content", os.path.basename(dest))
    parent, name = os.path.split(run_dir)
    if shutil.which("pigz"):
        subprocess.run(["tar", f"--use-compress-program=pigz -p {NCPU}", "-cf", tmp, "-C", parent, name], check=True)
    else:
        with tarfile.open(tmp, "w:gz") as t:
            t.add(run_dir, arcname=name)
    shutil.move(tmp, dest)

rows = load_table()
counts = {"ok": 0, "skipped": 0, "failed": 0}
BUSCO_RUNS = []
t_all = time.time()
for lin, info in LINEAGE_INFO.items():
    lt = safe(lin)
    for k in ("summaries", "full_tables", "archives"):
        os.makedirs(os.path.join(SUB[k], lt), exist_ok=True)
    pred = []
    if MODE == "genome" and info.get("domain") == "eukaryota":
        pred = ["--" + GENOME_PREDICTOR]
    for j in INPUTS:
        run = f"{j['sample']}_busco_{MODE_TAG}_{lt}"
        summ_json = os.path.join(SUB["summaries"], lt, f"short_summary.{run}.json")
        if SKIP_DONE and os.path.exists(summ_json):
            prev = rows.get((j["sample"], MODE, lin), {}).get("minutes", "NA")
            rows[(j["sample"], MODE, lin)] = row_from_json(summ_json, j["sample"], lin, run, j["src"], prev)
            counts["skipped"] += 1
            print(f"skip  {run} (already summarised)")
            continue

        out_parent = os.path.join(LOCAL_RUNS, lt)
        run_dir = os.path.join(out_parent, run)
        cmd = ["busco", "-i", j["path"], "-m", MODE, "-l", info["path"], "-o", run, "--out_path", out_parent,
               "-c", str(NCPU), "-f", "--offline", "--download_path", DL, "--opt-out-run-stats",
               *pred, *shlex.split(EXTRA_ARGS)]
        print(f"\n##### BUSCO {MODE}: {j['sample']}  x  {lin}  [{datetime.now():%H:%M:%S}] #####")
        local_log = os.path.join("/content", run + ".log")
        t0 = time.time()
        rc = run_logged(cmd, local_log)
        minutes = f"{(time.time() - t0) / 60:.1f}"
        shutil.copy(local_log, os.path.join(SUB["logs"], run + ".log"))

        js = glob.glob(os.path.join(run_dir, "short_summary.specific.*.json"))
        if rc == 0 and js:
            txt = js[0][:-5] + ".txt"
            shutil.copy(js[0], summ_json)
            if os.path.exists(txt):
                shutil.copy(txt, os.path.join(SUB["summaries"], lt, f"short_summary.{run}.txt"))
            ft = glob.glob(os.path.join(run_dir, "run_*", "full_table.tsv"))
            if ft:
                with open(ft[0], "rb") as fi, gzip.open(os.path.join(SUB["full_tables"], lt, f"{run}_full_table.tsv.gz"), "wb") as fo:
                    shutil.copyfileobj(fi, fo)
            archive(run_dir, os.path.join(SUB["archives"], lt, run + ".tar.gz"))
            rows[(j["sample"], MODE, lin)] = row_from_json(js[0], j["sample"], lin, run, j["src"], minutes)
            counts["ok"] += 1
            print(f"   -> {rows[(j['sample'], MODE, lin)]['one_line']}  ({minutes} min)")
            BUSCO_RUNS.append(run)
            if not KEEP_LOCAL_RUNS:
                shutil.rmtree(run_dir, ignore_errors=True)
        else:
            rows[(j["sample"], MODE, lin)] = {"sample": j["sample"], "mode": MODE, "lineage": lin,
                                              "status": "failed", "minutes": minutes, "run": run, "input": j["src"]}
            counts["failed"] += 1
            print(f"   !! FAILED (exit {rc}) - log: {os.path.join(SUB['logs'], run + '.log')}")
            if STOP_ON_ERROR:
                save_table(rows)
                raise RuntimeError(f"BUSCO failed on {run}; see the log above.")
        save_table(rows)
save_table(rows)

# provenance: one entry per invocation of this cell, appended
cfg_path = os.path.join(RUNDIR, "run_config.json")
cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {"run_id": RUN_ID, "invocations": []}
cfg["invocations"].append({
    "finished": datetime.now().isoformat(timespec="seconds"),
    "mode": MODE, "genome_predictor": GENOME_PREDICTOR if MODE == "genome" else None,
    "cpus": NCPU, "extra_args": EXTRA_ARGS, "tools": TOOL_VERSIONS,
    "lineages": {k: {x: v.get(x) for x in ("source", "domain", "creation_date", "number_of_BUSCOs",
                                           "number_of_species", "OrthoDB_version", "dataset_version")}
                 for k, v in LINEAGE_INFO.items()},
    "inputs": [{"sample": j["sample"], "src": j["src"], "n_seqs": j["n_seqs"], "mb": round(j["mb"], 2)} for j in INPUTS],
    "result": counts})
with open("/content/run_config.json", "w") as f:
    json.dump(cfg, f, indent=2)
shutil.copy("/content/run_config.json", cfg_path)

print(f"\nDone in {(time.time() - t_all) / 60:.1f} min: {counts['ok']} ok, {counts['skipped']} skipped, {counts['failed']} failed")
print("Summary table:", TABLE, f"({len(rows)} row(s))")

In [ ]:
#@title 8 · Summary table & plot { display-mode: "form" }
#@markdown Reads `busco_summary.tsv` (every mode and lineage run in this folder) and draws the standard BUSCO
#@markdown stacked bars: one panel per mode × lineage, saved to `plots/`.
SORT_BY = "input order"  #@param ["input order", "name", "completeness"]
PLOT_FORMATS = "png, pdf"  #@param ["png, pdf", "png", "pdf", "svg", "png, pdf, svg"]
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display

df = pd.read_csv(TABLE, sep="\t", dtype=str, keep_default_na=False)
num = ["C%", "S%", "D%", "F%", "M%"]
for c in num + ["n", "minutes"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

show = ["sample", "mode", "lineage", "status", "C%", "S%", "D%", "F%", "M%", "n", "minutes"]
display(df[show])
ok = df[df["status"] == "ok"].copy()
if (df["status"] != "ok").any():
    print("Failed:", ", ".join(df.loc[df["status"] != "ok", "run"]))
assert len(ok), "No successful runs to plot."

ok["panel"] = ok["mode"] + " · " + ok["lineage"]
print("\nComplete (C%) per sample:")
display(ok.pivot_table(index="sample", columns="panel", values="C%", aggfunc="first"))

order = [j["sample"] for j in INPUTS if j["sample"] in set(ok["sample"])] if "INPUTS" in dir() else []
order += sorted(set(ok["sample"]) - set(order))
if SORT_BY == "name":
    order = sorted(order)
elif SORT_BY == "completeness":
    order = list(ok.groupby("sample")["C%"].mean().sort_values(ascending=False).index)

COL = {"S%": "#56B4E9", "D%": "#3492C7", "F%": "#F0E442", "M%": "#F04442"}   # BUSCO's own palette
LAB = {"S%": "Complete (C) and single-copy (S)", "D%": "Complete (C) and duplicated (D)",
       "F%": "Fragmented (F)", "M%": "Missing (M)"}
panels = list(dict.fromkeys(ok.sort_values(["mode", "lineage"])["panel"]))
fig, axes = plt.subplots(1, len(panels), figsize=(4.8 * len(panels) + 2.2, 0.32 * len(order) + 1.6),
                         sharey=True, squeeze=False)
ypos = {s: i for i, s in enumerate(reversed(order))}
for ax, pan in zip(axes[0], panels):
    sub = ok[ok["panel"] == pan].set_index("sample")
    left = pd.Series(0.0, index=sub.index)
    y = [ypos[s] for s in sub.index]
    for k in ("S%", "D%", "F%", "M%"):
        ax.barh(y, sub[k], left=left, color=COL[k], height=0.75, edgecolor="white", linewidth=0.4)
        left += sub[k]
    for s, r in sub.iterrows():
        ax.text(1.5, ypos[s], f"C:{r['C%']:.1f}% [S:{r['S%']:.1f}%, D:{r['D%']:.1f}%], F:{r['F%']:.1f}%, M:{r['M%']:.1f}%",
                va="center", ha="left", fontsize=7, color="black")
    n = int(sub["n"].dropna().iloc[0]) if sub["n"].notna().any() else "?"
    ax.set_title(f"{pan}  (n = {n})", fontsize=9, loc="left")
    ax.set_xlim(0, 100); ax.set_xlabel("% BUSCOs")
    ax.spines[["top", "right"]].set_visible(False)
axes[0][0].set_yticks(range(len(order)))
axes[0][0].set_yticklabels(list(reversed(order)), fontsize=8)
fig.legend(handles=[Patch(color=COL[k], label=LAB[k]) for k in COL], loc="lower center",
           ncol=4, fontsize=8, frameon=False, bbox_to_anchor=(0.5, 0))
fig.tight_layout(rect=(0, 0.06 if len(order) > 4 else 0.12, 1, 1))

PLOT_DIR = os.path.join(RUNDIR, "plots"); os.makedirs(PLOT_DIR, exist_ok=True)
for ext in [e.strip() for e in PLOT_FORMATS.split(",")]:
    out = os.path.join(PLOT_DIR, f"busco_summary.{ext}")
    fig.savefig(out, dpi=300, bbox_inches="tight")
    print("saved:", out)
plt.show()

In [ ]:
#@title 9 · BUSCO status matrix across samples { display-mode: "form" }
#@markdown One matrix per mode × lineage: rows = BUSCO IDs, columns = samples, values `S` single-copy, `D` duplicated,
#@markdown `F` fragmented, `M` missing. Also lists the BUSCOs that are complete single-copy in at least `SCO_MIN_FRACTION`
#@markdown of the samples — a starting set for phylogenomics (sequences are in each archive under `run_*/busco_sequences/`).
SCO_MIN_FRACTION = 1.0  #@param {type:"slider", min:0.5, max:1.0, step:0.05}
import os, gzip, math
import pandas as pd
from IPython.display import display

CODE = {"Complete": "S", "Duplicated": "D", "Fragmented": "F", "Missing": "M"}
MAT_DIR = os.path.join(RUNDIR, "matrices"); os.makedirs(MAT_DIR, exist_ok=True)
tab = pd.read_csv(TABLE, sep="\t", dtype=str, keep_default_na=False)
tab = tab[tab["status"] == "ok"]

def read_full_table(path):
    status = {}
    with gzip.open(path, "rt") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            bid, st = line.rstrip("\n").split("\t")[:2]
            status.setdefault(bid, CODE.get(st, st))
    return status

MATRICES = {}
for (mode, lin), grp in tab.groupby(["mode", "lineage"], sort=True):
    lt = safe(lin) if "safe" in dir() else lin
    cols = {}
    for _, r in grp.iterrows():
        p = os.path.join(RUNDIR, "full_tables", lt, f"{r['run']}_full_table.tsv.gz")
        if os.path.exists(p):
            cols[r["sample"]] = read_full_table(p)
    if not cols:
        continue
    m = pd.DataFrame(cols).fillna("M")
    m.index.name = "busco_id"
    m = m.sort_index()
    tagm = {"proteins": "prot", "genome": "genome", "transcriptome": "tx"}[mode]
    m.to_csv(os.path.join(MAT_DIR, f"busco_matrix_{tagm}_{lt}.tsv"), sep="\t")
    need = math.ceil(SCO_MIN_FRACTION * m.shape[1] - 1e-9)
    sco = m.index[(m == "S").sum(axis=1) >= need].tolist()
    with open(os.path.join(MAT_DIR, f"sco_{tagm}_{lt}_min{SCO_MIN_FRACTION:.2f}.txt"), "w") as f:
        f.write("\n".join(sco) + ("\n" if sco else ""))
    MATRICES[(mode, lin)] = m
    print(f"\n=== {mode} · {lin}: {m.shape[0]} BUSCOs x {m.shape[1]} samples ===")
    display(m.apply(lambda c: c.value_counts()).reindex(["S", "D", "F", "M"]).fillna(0).astype(int))
    print(f"single-copy in >= {need}/{m.shape[1]} samples: {len(sco)} BUSCOs")
    missing_all = int((m == "M").all(axis=1).sum())
    if m.shape[1] > 1:
        print(f"missing in every sample : {missing_all}  (shared absences - clade-level loss or a common gap)")
print("\nMatrices and SCO lists in:", MAT_DIR)

In [ ]:
#@title 10 · Download results { display-mode: "form" }
#@markdown With Drive mounted everything is already in the run folder. Without Drive (or to grab a copy) this zips the
#@markdown run folder and downloads it; `summary only` leaves out the per-run `.tar.gz` archives.
DOWNLOAD = "summary only"  #@param ["summary only", "everything", "nothing (just list files)"]
import os, zipfile

total = 0
print(RUNDIR)
for root, dirs, files_ in sorted(os.walk(RUNDIR)):
    dirs.sort()
    rel = os.path.relpath(root, RUNDIR)
    size = sum(os.path.getsize(os.path.join(root, f)) for f in files_)
    total += size
    if files_:
        print(f"  {'.' if rel == '.' else rel + '/':<45}{len(files_):>5} file(s) {size / 1e6:9.1f} MB")
print(f"  {'total':<45}{'':>14}{total / 1e6:9.1f} MB")

if DOWNLOAD != "nothing (just list files)":
    zpath = f"/content/{RUN_ID}{'_summary' if DOWNLOAD == 'summary only' else ''}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files_ in os.walk(RUNDIR):
            rel = os.path.relpath(root, RUNDIR)
            if DOWNLOAD == "summary only" and rel.split(os.sep)[0] == "archives":
                continue
            for f in files_:
                p = os.path.join(root, f)
                z.write(p, os.path.join(RUN_ID, os.path.relpath(p, RUNDIR)))
    print(f"\n{zpath}: {os.path.getsize(zpath) / 1e6:.1f} MB")
    from google.colab import files
    files.download(zpath)

---
### Notes & troubleshooting
- **Outputs** land in a timestamped run folder, `WORKDIR/runs/run_<date>-<time>[_tag]/`:
  `busco_summary.tsv` (one row per sample × mode × lineage: C/S/D/F/M percentages and counts, internal-stop-codon %, assembly metrics in genome mode, runtime, status), `summaries/<lineage>/` (each run's `short_summary` `.txt` + `.json`), `full_tables/<lineage>/` (gzipped `full_table.tsv`), `archives/<lineage>/` (whole BUSCO directory per run, `.tar.gz`), `logs/`, `plots/`, `matrices/` and `run_config.json` (tool versions, lineage dataset dates, inputs and parameters for every invocation). Lineage datasets live outside the run folders in `WORKDIR/busco_lineages/` and are downloaded once.
- **Run names** follow `<sample>_busco_<prot|genome|tx>_<lineage>`, so several modes and lineages share one run folder without collisions.
- **Adding lineages or modes:** change `LINEAGES` / `MODE` in cell 4 and re-run cells 4-9; rows accumulate in the same `busco_summary.tsv`, keyed by (sample, mode, lineage) — re-running a combination replaces its row instead of duplicating it.
- **Resuming a batch after a disconnect:** in the new session set `RESUME_RUN` in cell 3 to the interrupted folder's name and run cells 1-7 again; with `SKIP_DONE` on, runs that already have a summary are skipped.
- **Failed samples** get a `status = failed` row with `NA` scores and the batch carries on (unless `STOP_ON_ERROR`); the full BUSCO output is in `logs/<run>.log`.
- **Choosing a lineage:** use the most specific dataset that contains all samples (e.g. `oomycota_odb12.2` for oomycetes, with `stramenopiles_odb12.2` as a broader check). Both `<name>_odb12` and `<name>_odb12.2` exist on the server (`odb12.2` needs BUSCO >= 6.1); use the same release across a comparison — scores from different dataset versions are not directly comparable.
- **HPC lineage folders:** copy the lineage directories (e.g. from the cluster's `busco_lineages/`) to Drive and set `LINEAGE_DIR`, or type a full path in `LINEAGES`.
- **Proteins mode** expects one isoform per gene for clean duplication scores; with all isoforms in the proteome, alternative transcripts inflate `D%`.
- **Genome mode** defaults to **miniprot** for eukaryotes (BUSCO's default since v5.7); MetaEuk is the slower alternative. `stop_codon%` is the fraction of complete miniprot matches with internal stop codons — high values flag frameshifts or pseudogenes. Prokaryote lineages use Prodigal automatically.
- **BUSCO + Drive:** BUSCO writes symlinks, which the Drive mount does not support, so it always runs locally; only the copies and archives go to Drive.
- **Speed:** proteins mode takes minutes; genome/transcriptome mode on whole plant genomes can take hours on 2 free-tier cores — fine for small genomes and drafts, better on HPC at scale.
- **Edit a hidden cell:** menu (vertical dots) > *Show code*.
- Cite: Tegenfeldt *et al.* (2025) *Nucleic Acids Res* 53(D1):D516 (OrthoDB v12); Manni *et al.* (2021) *Mol Biol Evol* 38:4647; Li (2023) *Bioinformatics* 39:btad014 (miniprot); Levy Karin *et al.* (2020) *Microbiome* 8:48 (MetaEuk).